# Do we reply faster when the conversation is about football? (week 5, relationships)

Builds the Analysis 18 chart from `analysis-log.md`.

**Decisions from stage 1 (student):**

- Proposition: conversations mostly about football (≥ 25% of messages match the keyword list) have replies about
  **20% faster** than conversations with no football. Falsified if there is no negative relation between reply time
  and football share.
- Conversation: a gap > 401 min starts a new one (analysis 15). Reply speed per conversation = median reply gap.
- Keyword list v2 below; countries dropped (holidays/news), Twente and Barcelona only with "FC".

This first part only checks the keyword list: how often each term hits, and two samples for the student to label.

In [ ]:
import re

import numpy as np
import pandas as pd

from wa_analyzer.data import load_own_chat

SEED = 42
CLUBS = [
    "fc twente", "ajax", "psv", "feyenoord", "heracles", "psg", "manchester city", "arsenal",
    "manchester united", "tottenham", "spurs", "juventus", "fc barcelona", "real madrid",
]
WORDS = [
    "voetbal", "wedstrijd", "football", "goals?", "kaart", "overtreding", "oranje", "ek", "wk", "eredivisie",
    "champions league", "penalty", "scheids", "scheidsrechter", "doelpunt", "elftal", "buitenspel", "var",
    "spits", "topspits",  # from label check 1
    "keeper", "wereldkampioen", "qatar", "scoren", "scoort", "scoorde", "gescoord",  # from label check 3
]
PLAYERS = [  # the four the student named, then other popular players (assistant's additions, to review)
    "ronaldo", "messi", "mbapp[eé]", "neymar",
    "haaland", "depay", "gakpo", "salah", "kane", "lewandowski", "modri[cć]", "de bruyne", "bellingham", "yamal",
    "schouten", "hooiveld", "weghorst", "bergwijn", "van persie",  # from label check 1; "van dijk" dropped (surname in the chat)
]
COUNTRIES = [  # dropped in v2 (holidays/news), back in v4 after label check 1 found them in missed match talk
    "nederland", "frankrijk", "argentini[eë]", "engeland", "engelse", "spanje", "senegal",
]
NICKNAMES = ["wout", "woutje"]  # Weghorst; nobody in the network is called Wout (student)
FOOTBALL = re.compile(r"\b(" + "|".join(CLUBS + WORDS + PLAYERS + COUNTRIES + NICKNAMES) + r")\b", re.IGNORECASE)
SCORE = re.compile(r"(?<![\d:/-])\b\d-\d\b(?![\d:/-])")  # "2-0", not "12-06", "20:45" or "1-6-2024"; ranges like "2-3 dagen" still hit
GLUED_SYSTEM_LINE = re.compile(r"\d{2}-\d{2}-\d{4} \d{2}:\d{2} - ")  # parser glitch; carries real names
NOT_A_TEXT = re.compile(r"<Media weggelaten>|bericht is verwijderd|locatie gedeeld|^locatie: |\.vcf \(bestand")
NL_MATCH_DAYS = pd.to_datetime([  # analysis 6 / notebook 04
    "2021-06-13", "2021-06-17", "2021-06-21", "2021-06-27",
    "2022-11-21", "2022-11-25", "2022-11-29", "2022-12-03", "2022-12-09",
    "2024-06-16", "2024-06-21", "2024-06-25", "2024-07-02", "2024-07-06", "2024-07-10",
    "2026-06-14", "2026-06-20", "2026-06-25", "2026-06-30",
])

PAYMENT_REQUEST = re.compile(r"betaalverzoek|payreq", re.IGNORECASE)  # not conversation (label check 3)

own = load_own_chat()
# same fix as 06-13: `timestamp` is stored as strings; the `+00:00` is only a label
own["timestamp"] = pd.to_datetime(own["timestamp"], utc=True).dt.tz_localize(None)
own = own[~own["message"].fillna("").astype(object).str.contains(PAYMENT_REQUEST)]
own = own.sort_values("timestamp", kind="stable").reset_index(drop=True)
own["day"] = own["timestamp"].dt.normalize()
text = own["message"].fillna("").astype(object)
own["football_term"] = text.str.extract(FOOTBALL, expand=False).str.lower().where(lambda term: term.notna(), text.str.extract(f"({SCORE.pattern})", expand=False).where(lambda s: s.isna(), "score"))
own["is_football"] = own["football_term"].notna()
own["glued_system_line"] = text.str.contains(GLUED_SYSTEM_LINE)
own["not_a_text"] = text.str.contains(NOT_A_TEXT)
print(f"{own['is_football'].sum():,} of {len(own):,} messages match ({own['is_football'].mean():.1%})")

In [ ]:
# which terms carry the matches -- a term with many hits and many false hits matters most
own["football_term"].value_counts().head(25)

## Label check (student)

Two samples written to `data/processed/football-label-check.csv` (gitignored, never committed).
Fill the `is_football` column with `y` or `n` for each row.

- **matched**: 50 random messages the list matches → share of `n` = false hits.
- **match_day_unmatched**: 50 random text messages on NL match days the list does *not* match → share of `y` = misses.

In [ ]:
rng = np.random.default_rng(SEED)
labelable = own[~own["glued_system_line"] & ~own["not_a_text"]]
matched = labelable[labelable["is_football"]]
match_day_unmatched = labelable[~labelable["is_football"] & labelable["day"].isin(NL_MATCH_DAYS)]
sample = pd.concat(
    [
        matched.sample(50, random_state=SEED).assign(sample="matched"),
        match_day_unmatched.sample(50, random_state=SEED).assign(sample="match_day_unmatched"),
    ]
)[["sample", "timestamp", "football_term", "message"]].assign(is_football="")
sample.to_csv("../../data/processed/football-label-check.csv", index=False)
print(f"pool sizes: matched {len(matched):,}, match-day unmatched {len(match_day_unmatched):,}")

## Label check 4 (list v5, fresh sample, last round)

List v5 = v4 + keeper, wereldkampioen, Qatar, scoren/scoort/scoorde/gescoord, Van Persie; payment requests dropped from the
data. Fresh messages only: rows in `football-label-check.csv` and `football-label-check-3.csv` are left out.
(`football-label-check-2.csv` was never labelled.)

In [ ]:
already_labelled = pd.concat(
    pd.read_csv(f"../../data/processed/{name}", parse_dates=["timestamp"])
    for name in ["football-label-check.csv", "football-label-check-3.csv"]
)
seen = set(zip(already_labelled["timestamp"], already_labelled["message"]))
fresh = labelable[[(t, m) not in seen for t, m in zip(labelable["timestamp"], labelable["message"])]]
fresh_matched = fresh[fresh["is_football"]]
fresh_unmatched = fresh[~fresh["is_football"] & fresh["day"].isin(NL_MATCH_DAYS)]
sample_4 = pd.concat(
    [
        fresh_matched.sample(50, random_state=SEED + 3).assign(sample="matched"),
        fresh_unmatched.sample(50, random_state=SEED + 3).assign(sample="match_day_unmatched"),
    ]
)[["sample", "timestamp", "football_term", "message"]].assign(is_football="")
sample_4.to_csv("../../data/processed/football-label-check-4.csv", index=False)
print(f"fresh pools: matched {len(fresh_matched):,}, match-day unmatched {len(fresh_unmatched):,}")
print(f"match days: {labelable['day'].isin(NL_MATCH_DAYS).sum()} text messages, {labelable.loc[labelable['day'].isin(NL_MATCH_DAYS), 'is_football'].sum()} matched")

In [ ]:
# hits per country -- the new terms most at risk of holiday/news false hits
own.loc[own["football_term"].str.match("|".join(COUNTRIES), na=False), "football_term"].value_counts()

## Label check 5 (list v6: scores and nickname only)

v6 = v5 + scores ("2-0") and "Wout(je)". Only these new hits are checked: scores can also be ranges ("2-3 dagen") or dates.

In [ ]:
new_terms = labelable.assign(football_term=own["football_term"])
new_terms = new_terms[new_terms["football_term"].isin(["score", "wout", "woutje"])]
sample_5 = new_terms.sample(min(40, len(new_terms)), random_state=SEED + 4).assign(sample="score_or_nickname")
sample_5[["sample", "timestamp", "football_term", "message"]].assign(is_football="").to_csv(
    "../../data/processed/football-label-check-5.csv", index=False
)
print(f"new hits from scores/nickname only: {len(new_terms)}, sampled {len(sample_5)}")

## Conversations and reply times

List v6 is final. Conversation: a gap > 401 min starts a new one; kept if ≥ 5 messages and at least one reply.
Reply and jitter as in analyses 15–16: a message directly after someone else's message, gap in minutes with the corrected jitter
for whole-minute timestamps. One value per conversation: its median reply gap.
Groups: 0 / 1 / ≥ 2 football messages.

**Rules, fixed before running** (from stage 1, expectation ≥ 10% faster):

| Check | Changes the conclusion if |
|---|---|
| Main: group medians, 95% range from resampling days | the ≥ 2 median is not lower than the 0 median, or its range overlaps the 0 median |
| Length: same chart within 5–10 / 11–25 / 26+ messages | the ≥ 2 group is not faster than 0 in most length bands |
| Continuous: football share vs median reply time | the smoothed line does not go down |

In [ ]:
import matplotlib.pyplot as plt
from statsmodels.nonparametric.smoothers_lowess import lowess

CUT = 401  # minutes -- conversation cut from analysis 15
MIN_MESSAGES = 5
GROUP_ORDER = ["0", "1", "2+"]
GROUP_COLOR = "#2a78d6"
DOT_STYLE = {"0": ("#2a78d6", 0.18), "1": ("#eb6834", 0.6), "2+": ("#eb6834", 0.6)}  # football groups: fewer dots, so stronger

rng = np.random.default_rng(SEED)
recorded = own["timestamp"].diff().dt.total_seconds() / 60
s1, s2 = rng.uniform(0, 1, len(own)), rng.uniform(0, 1, len(own))
own["gap"] = np.where(recorded == 0, np.abs(s2 - s1), recorded + s2 - s1)
own["conversation"] = (recorded.isna() | (recorded > CUT)).cumsum()
own["is_reply"] = (
    own["author"].ne(own["author"].shift())
    & own["conversation"].eq(own["conversation"].shift())
    & (own["gap"] > 0)
)

conversations = own.groupby("conversation").agg(
    day=("day", "first"),
    messages=("is_football", "size"),
    football_messages=("is_football", "sum"),
)
conversations["median_reply"] = own[own["is_reply"]].groupby("conversation")["gap"].median()
conversations = conversations[(conversations["messages"] >= MIN_MESSAGES) & conversations["median_reply"].notna()].copy()
conversations["football_share"] = conversations["football_messages"] / conversations["messages"]
conversations["group"] = pd.cut(conversations["football_messages"], [-1, 0, 1, np.inf], labels=GROUP_ORDER)
conversations.groupby("group", observed=True).agg(
    conversations=("messages", "size"), median_length=("messages", "median"), median_reply=("median_reply", "median")
).round(2)

In [ ]:
def group_medians(frame):
    return frame.groupby("group", observed=True)["median_reply"].median().reindex(GROUP_ORDER)


def day_resampled_ranges(frame, draws=1000):
    by_day = dict(tuple(frame.groupby("day")))
    days = np.array(list(by_day))
    medians = pd.DataFrame([group_medians(pd.concat([by_day[d] for d in rng.choice(days, len(days))])) for _ in range(draws)])
    return medians.quantile([0.025, 0.975]).T.set_axis(["lo", "hi"], axis=1)


main = pd.concat([group_medians(conversations).rename("median"), day_resampled_ranges(conversations)], axis=1)
main["vs_0"] = main["median"] / main.loc["0", "median"] - 1
main.round(2)

In [ ]:
def dots_and_medians(ax, frame, summary):
    for position, group in enumerate(GROUP_ORDER):
        values = frame.loc[frame["group"] == group, "median_reply"]
        color, alpha = DOT_STYLE[group]
        ax.scatter(position + rng.uniform(-0.25, 0.25, len(values)), values, s=10, color=color, alpha=alpha, linewidths=0)
    ax.vlines(range(3), summary["lo"], summary["hi"], color="0.15", linewidth=2)
    ax.plot(range(3), summary["median"], color="0.15", marker="o", markersize=7, linewidth=1.5)
    counts = frame["group"].value_counts().reindex(GROUP_ORDER)
    ax.set_xticks(range(3), [f"{group}\n(n = {counts[group]})" for group in GROUP_ORDER])
    ax.set_yscale("log")
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(axis="y", color="0.9")


fig, ax = plt.subplots(figsize=(7, 5.5))
dots_and_medians(ax, conversations, main)
ax.set_xlabel("football messages in the conversation")
ax.set_ylabel("median reply time in the conversation (min, log scale)")
ax.set_title("Draft: reply time by football in the conversation\n(dot = conversation; black = median, 95% range from resampling days)", loc="left", fontsize=10)
fig.tight_layout()
fig.savefig("football-reply-draft.png", dpi=150)

### Check: conversation length

In [ ]:
LENGTH_BANDS = [(5, 10), (11, 25), (26, np.inf)]
fig, axes = plt.subplots(1, 3, figsize=(13, 5), sharey=True)
length_rows = []
for ax, (low, high) in zip(axes, LENGTH_BANDS):
    band = conversations[conversations["messages"].between(low, high)]
    summary = pd.concat([group_medians(band).rename("median"), day_resampled_ranges(band)], axis=1)
    dots_and_medians(ax, band, summary)
    label = f"{low}-{high:.0f} messages" if np.isfinite(high) else f"{low}+ messages"
    ax.set_title(label, loc="left")
    length_rows.append(summary.assign(band=label))
axes[0].set_ylabel("median reply time (min, log scale)")
fig.tight_layout()
fig.savefig("football-reply-length-check.png", dpi=150)
pd.concat(length_rows).reset_index(names="group").pivot(index="band", columns="group", values="median").reindex(columns=GROUP_ORDER).round(2)

### Check: football share (continuous)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(conversations["football_share"] * 100, conversations["median_reply"], s=8, color=GROUP_COLOR, alpha=0.25, linewidths=0)
smooth = lowess(np.log10(conversations["median_reply"]), conversations["football_share"] * 100, frac=0.5)
ax.plot(smooth[:, 0], 10 ** smooth[:, 1], color="0.15", linewidth=2)
ax.set_yscale("log")
ax.set_xlabel("share of football messages in the conversation (%)")
ax.set_ylabel("median reply time (min, log scale)")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("football-reply-share-check.png", dpi=150)
print(f"conversations at 0%: {(conversations['football_share'] == 0).mean():.0%}")

### How much faster once length is taken into account?

Straight-line fit on the log scale: log(median reply) ~ group + log(conversation length). The group coefficient then compares
conversations **of the same length**. 95% range from resampling days.

In [ ]:
def length_adjusted(frame):
    design = np.column_stack([
        np.ones(len(frame)),
        (frame["group"] == "1").to_numpy(float),
        (frame["group"] == "2+").to_numpy(float),
        np.log10(frame["messages"]),
    ])
    coefficients, *_ = np.linalg.lstsq(design, np.log10(frame["median_reply"]), rcond=None)
    return pd.Series(10 ** coefficients[1:3] - 1, index=["1", "2+"])


by_day = dict(tuple(conversations.groupby("day")))
days = np.array(list(by_day))
adjusted_draws = pd.DataFrame([length_adjusted(pd.concat([by_day[d] for d in rng.choice(days, len(days))])) for _ in range(1000)])
pd.DataFrame({
    "unadjusted_vs_0": main.loc[["1", "2+"], "vs_0"],
    "same_length_vs_0": length_adjusted(conversations),
    "lo": adjusted_draws.quantile(0.025),
    "hi": adjusted_draws.quantile(0.975),
}).round(2)

In [ ]:
# which football terms carry the 11-25 message, 2+ conversations (counts only, no message text)
middle = conversations[conversations["messages"].between(11, 25) & (conversations["group"] == "2+")].index
own.loc[own["conversation"].isin(middle), "football_term"].value_counts().head(12)

## Stage 6 — checks on the same-length effect (rules fixed before running)

All on the same-length comparison (2+ vs 0, controlling for log conversation length).

| Check | Changes the claim if |
|---|---|
| Without NL match days / without tournament periods ("in general") | 2+ is no longer faster, or its range includes 0 |
| Null: shuffle groups within length deciles, 1,000× | ≥ 5% of shuffles are as fast as the real effect |
| + time of day (night / morning / afternoon / evening) and weekend | the 2+ effect shrinks by more than half |
| Held-out years: 2020–22 vs 2023–26 | the direction flips in either period |

In [ ]:
TOURNAMENTS = [("2021-06-11", "2021-07-11"), ("2022-11-20", "2022-12-18"), ("2024-06-14", "2024-07-14"), ("2026-06-11", "2026-07-19")]
start = own.groupby("conversation")["timestamp"].first()
conversations["start_hour"] = start.reindex(conversations.index).dt.hour
conversations["weekend"] = start.reindex(conversations.index).dt.dayofweek >= 5
conversations["in_tournament"] = np.logical_or.reduce(
    [conversations["day"].between(pd.Timestamp(a), pd.Timestamp(b)) for a, b in TOURNAMENTS]
)
conversations["nl_match_day"] = conversations["day"].isin(NL_MATCH_DAYS)
conversations["part_of_day"] = pd.cut(conversations["start_hour"], [-1, 5, 11, 17, 23], labels=["night", "morning", "afternoon", "evening"])


def effect_2plus(frame, extra=()):
    """2+ vs 0 at the same conversation length (and the extra columns), as a relative change."""
    columns = [np.ones(len(frame)), (frame["group"] == "1").to_numpy(float), (frame["group"] == "2+").to_numpy(float), np.log10(frame["messages"])]
    for column in extra:
        dummies = pd.get_dummies(frame[column], drop_first=True, dtype=float)
        columns += [dummies[c].to_numpy() for c in dummies]
    coefficients, *_ = np.linalg.lstsq(np.column_stack(columns), np.log10(frame["median_reply"]), rcond=None)
    return 10 ** coefficients[2] - 1


def with_range(frame, label, extra=(), draws=1000):
    by_day = dict(tuple(frame.groupby("day")))
    days = np.array(list(by_day))
    draws_ = [effect_2plus(pd.concat([by_day[d] for d in rng.choice(days, len(days))]), extra) for _ in range(draws)]
    lo, hi = np.percentile(draws_, [2.5, 97.5])
    return {"version": label, "2+ conversations": int((frame["group"] == "2+").sum()), "2+ vs 0": effect_2plus(frame, extra), "lo": lo, "hi": hi}


checks = pd.DataFrame([
    with_range(conversations, "main (same length)"),
    with_range(conversations[~conversations["nl_match_day"]], "without NL match days"),
    with_range(conversations[~conversations["in_tournament"]], "without tournament periods"),
    with_range(conversations, "+ time of day and weekend", extra=("part_of_day", "weekend")),
    with_range(conversations[conversations["day"] < "2023-01-01"], "2020-22 only"),
    with_range(conversations[conversations["day"] >= "2023-01-01"], "2023-26 only"),
])
checks.round(2)

In [ ]:
# null: shuffle the groups within length deciles, so length stays tied to reply time but football does not
deciles = pd.qcut(conversations["messages"], 10, duplicates="drop")
real = effect_2plus(conversations)
shuffled = []
for _ in range(1000):
    shuffled_groups = conversations.groupby(deciles, observed=True)["group"].transform(lambda g: g.sample(frac=1, random_state=rng.integers(1 << 31)).to_numpy())
    shuffled.append(effect_2plus(conversations.assign(group=shuffled_groups)))
shuffled = np.array(shuffled)
print(f"real 2+ vs 0: {real:.0%}; shuffled range {shuffled.min():.0%} to {shuffled.max():.0%}; as fast or faster: {(shuffled <= real).sum()}/1000")

## Final chart

Minutes on the y-axis, medians labelled; title (student's choice), subtitle and footnote agreed with the student.

In [ ]:
from matplotlib.ticker import FixedLocator, FuncFormatter, NullLocator

MINUTE_TICKS = [0.25, 0.5, 1, 2, 5, 10, 30, 60, 120, 240]


def minutes_label(value, _):
    if value < 1:
        return f"{value * 60:.0f} s"
    return f"{value / 60:.0f} h" if value >= 60 else f"{value:.0f} min"


fig, ax = plt.subplots(figsize=(8.5, 7.5))
dots_and_medians(ax, conversations, main)
for position, group in enumerate(GROUP_ORDER):
    ax.annotate(f"{main.loc[group, 'median']:.1f} min", (position, main.loc[group, "median"]), xytext=(12, 0),
                textcoords="offset points", va="center", fontsize=10, fontweight="bold", color="0.15",
                bbox={"boxstyle": "round,pad=0.2", "facecolor": "white", "edgecolor": "none"})
ax.yaxis.set_major_locator(FixedLocator(MINUTE_TICKS))
ax.yaxis.set_minor_locator(NullLocator())
ax.yaxis.set_major_formatter(FuncFormatter(minutes_label))
ax.set_ylim(0.1, 400)
ax.set_xlabel("football messages in the conversation")
ax.set_ylabel("Median reply time in conversation")
ax.set_title("Kick-off: football conversations move faster", loc="left", fontsize=13, pad=30)
ax.text(0, 1.02, "Typical reply in conversations with 2+ football messages: 1.6 min vs 8.6 min without football;\n"
        "about a third faster at the same conversation length", transform=ax.transAxes, fontsize=9, color="0.3", va="bottom")
FOOTNOTE = (
    "Dot = one conversation; black point = median of the group, bar = 95% range of that median (resampling days).\n"
    "Data: 1,052 conversations of 5+ messages (new conversation after 401 min of silence), group chat 2020-2026, 9 people.\n"
    "Football message = matches a list of clubs, players, countries, football words and scores.\n"
    "Limits: the list finds ~60% of football talk and ~96% of its hits are football; misses land in '0', which shrinks the gap.\n"
    "A single football message ('1') is not clearly faster.\n"
    "Checks, same conversation length: 2+ is ~40% faster (95% range 14-58%); without NL match days 42%, without tournaments 51%;\n"
    "1 in 1,000 shuffles as fast; with time of day and weekend 28%; weaker and uncertain in 2023-26 (19%)."
)
fig.text(0.01, 0.01, FOOTNOTE, fontsize=7, color="0.35", va="bottom")
fig.tight_layout(rect=(0, 0.19, 1, 1))
fig.savefig("football-reply-final.png", dpi=150)
